In [3]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import detectors as det
import filters as filt
import integrators as integ

csv_file_name = 'closed_loop_18-08-2026_20-23-47_1_CW_18m.csv'
csv_path = '../data/measured_closed_loops/'
csv_save_path = '../data/ZARU_and_loop_closure/'
df = pd.read_csv(f'{csv_path}{csv_file_name}', skipinitialspace=True)

In [ ]:
# ZARU WAS DEEMED REDUNDENT FOR THIS PROJECT - YIELDS WORSE RESULTS THAN MAHONY GZ CORRECTION.

# loop_logs = [
#     'closed_loop_18-08-2026_21-18-50_1_CW_NoStop_18m.csv',
#     'closed_loop_18-08-2026_21-19-41_2_CW_NoStop_18m.csv',
#     'closed_loop_18-08-2026_21-20-55_3_CW_NoStop_18m.csv',
# ]

# loop_logs = [
#     'closed_loop_18-08-2026_20-23-47_1_CW_18m.csv',
#     'closed_loop_18-08-2026_20-42-32_2_CW_18m.csv',
#     'closed_loop_18-08-2026_20-44-45_3_CW_18m.csv',
#     'closed_loop_18-08-2026_20-51-07_4_CW_18m.csv',
# ]

# loop_logs = [
#     'closed_loop_19-08-2026_21-17-41_1_CCW_16m_2min_walk_NO_Stops.csv',
#     'closed_loop_19-08-2026_21-17-41_1_CCW_16m_2min_walk_with_stops.csv',
# ]

loop_logs = [
    'closed_loop_18-08-2026_21-15-02_long_loop_CCW.csv',
]

csv_path = '../data/measured_closed_loops/'
ZARU_THRESHOLD = 2.0
ZARU_DWELL = 100

for file in loop_logs:
    df = pd.read_csv(f'{csv_path}{file}', skipinitialspace=True)
    zvw_mask = det.detect_zvw(df)

    gx, gy, gz = df['gx'].values, df['gy'].values, df['gz'].values
    ax, ay, az = df['ax'].values, df['ay'].values, df['az'].values
    omega_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    
    time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6
    print(f'\nProcessing: {file}')
    print(f'Total time: {time_sec.iloc[-1]:.2f} (s)')

    quiet_samples = (omega_mag < ZARU_THRESHOLD) & zvw_mask
    blocks = (quiet_samples != pd.Series(quiet_samples).shift()).cumsum()
    zaru_mask = ((pd.Series(quiet_samples).groupby(blocks).transform('sum') >= ZARU_DWELL) & quiet_samples).values

    print(f'Amount of ZARU windows detected: {len(np.unique(blocks[zaru_mask]))}')
    gz_cleaned, final_bias_z = filt.apply_zaru_single_mean(gz, zaru_mask)

    true_bias = np.mean(gz[:700])
    print(f'True bias (based on the first initial pause: {true_bias:.3f} °/s)')

    gz_use = gz_cleaned
    if zaru_mask.sum() > 0:
        assert not np.array_equal(gz_use, df['gz'].values), "ZARU active but raw gz reached the filter"

    dt_array = np.diff(df['t_us'] - df['t_us'].iloc[0])
    dt_array = np.insert(dt_array, 0, dt_array.mean()) / 1e6

    masked_quats = filt.mahony_filter(ax, ay, az, 
                                      gx, gy, gz_use, 
                                      dt_array, zvw_mask, Kp=2.0, Ki=0.5)

    raw_accel = np.column_stack((ax, ay, az))
    global_accel = filt.rotate_vector_by_quaternion(raw_accel, masked_quats)
    linear_accel = np.copy(global_accel)
    linear_accel[:, 2] -= 1.0 
    accel_ms2 = linear_accel * 9.80665 

    vel_zupt, pos_zupt, _ = integ.integrate_kinematics(accel_ms2, dt_array, zvw_mask)

    start_pos = pos_zupt[0, :2]
    end_pos = pos_zupt[-1, :2]
    closure_gap = np.linalg.norm(end_pos - start_pos)

    print(f"  -> ZARU Bias Estimate: {final_bias_z:.4f} °/s")
    print(f"  -> Closure Gap: {closure_gap:.3f} m")



Processing: closed_loop_18-08-2026_21-15-02_long_loop_CCW.csv
Total time: 53.40 (s)
Amount of ZARU windows detected: 5
True bias (based on the first initial pause: -0.307 °/s)
  -> ZARU Bias Estimate: -0.3098 °/s
  -> Closure Gap: 0.773 m
